# 03 — Exploratory Data Analysis (EDA)
## Hospital Readmission Risk Prediction System (DISCHARGE)
**Dataset:** UCI Diabetes 130-US Hospitals (1999–2008) · Cleaned Staging Encounters (N = 99,340)  
**Roadmap Tasks:** `P3-01` (Univariate), `P3-02` (Categorical vs Target), `P3-03` (Numeric vs Target), `P3-05` (Diagnoses), `P3-06` (Spearman Correlation)  
**Objective:** Uncover clinical patterns, feature distributions, and bivariate relationships with 30-day unplanned readmission (`readmitted_30d`), and save high-resolution figures for reports and presentations.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

# Matplotlib headless configuration for Windows (G-012)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from readmission.config import (
    REPO_ROOT,
    REPORTS_FIGURES_DIR,
    RANDOM_STATE,
)
from readmission.data.load import load_staging_encounters
from readmission.features.icd9 import add_diagnosis_groups

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'axes.edgecolor': '#CBD5E1',
    'grid.color': '#F1F5F9',
    'grid.linestyle': '--',
    'figure.autolayout': True,
})

print('Libraries imported successfully.')

### 1. Load Clean Staging Data
We pull the 99,340 verified clinical encounters from `staging.encounters` (Neon PostgreSQL) and classify ICD-9 codes into the Strack et al. (2014) high-level clinical categories using `src/readmission/features/icd9.py`.

In [ ]:
df = load_staging_encounters()
print(f'Loaded staging encounters: {df.shape[0]:,} rows x {df.shape[1]} columns')
df = add_diagnosis_groups(df)
print('ICD-9 diagnosis grouping applied: diag_1_group, diag_2_group, diag_3_group ready.')
df[['encounter_id', 'patient_nbr', 'race', 'gender', 'age_bucket', 'diag_1', 'diag_1_group', 'readmitted_30d']].head()

### 2. Target Distribution & Imbalance (`P3-01`)
Evaluate the primary prediction target: 30-day unplanned readmission (`readmitted_30d = 1`).

In [ ]:
target_counts = df['readmitted_30d'].value_counts().sort_index()
base_rate = target_counts[1] / len(df) * 100
print(f'Class 0 (Not Readmitted <30d): {target_counts[0]:,} ({100-base_rate:.2f}%)')
print(f'Class 1 (Readmitted <30d):     {target_counts[1]:,} ({base_rate:.2f}%)')
print(f'Total Clinical Encounters:    {len(df):,}')
print(f'Unique Patients:              {df["patient_nbr"].nunique():,}')

### 3. Patient Demographics & Utilization Profiles
Analyze patient age brackets, gender distribution, and racial cohorts across the inpatient encounters.

In [ ]:
demo_summary = pd.DataFrame({
    'Encounters': df['age_bucket'].value_counts().sort_index(),
    'Share (%)': (df['age_bucket'].value_counts().sort_index() / len(df) * 100).round(2),
    'Readmission Rate (%)': (df.groupby('age_bucket')['readmitted_30d'].mean() * 100).round(2)
})
demo_summary

### 4. Continuous Feature Summary & Skewness (`P3-03`)
Review median, IQR, and dispersion metrics for length of stay, procedures, medications, and prior hospital visits.

In [ ]:
num_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_diagnoses', 'number_inpatient', 'number_emergency', 'number_outpatient']
num_stats = df[num_cols].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.99]).T
num_stats['IQR'] = num_stats['75%'] - num_stats['25%']
num_stats[['mean', 'std', 'min', '50%', 'IQR', '90%', '99%', 'max']]

### 5. Categorical Readmission Rates with 95% Wilson Confidence Intervals (`P3-02`)
Assess readmission risk variation across key clinical categories: HbA1c result, insulin regimen, and diabetic med modification.

In [ ]:
for col in ['a1c_result', 'max_glu_serum', 'insulin', 'change']:
    grp = df.groupby(col)['readmitted_30d'].agg(['count', 'sum', 'mean'])
    grp['readmit_rate_%'] = (grp['mean'] * 100).round(2)
    print(f'\n--- {col} ---')
    print(grp[['count', 'sum', 'readmit_rate_%']])

### 6. Primary ICD-9 Diagnosis Group Breakdown (`P3-05`)
Evaluate readmission rates and encounter volumes across Strack's 9 clinical ICD-9 diagnostic groups.

In [ ]:
diag_grp = df.groupby('diag_1_group')['readmitted_30d'].agg(['count', 'sum', 'mean'])
diag_grp['share_%'] = (diag_grp['count'] / len(df) * 100).round(2)
diag_grp['readmit_rate_%'] = (diag_grp['mean'] * 100).round(2)
diag_grp = diag_grp.sort_values('count', ascending=False)
diag_grp[['count', 'share_%', 'readmit_rate_%']]

### 7. Spearman Correlation & Multicollinearity Assessment (`P3-06`)
Examine non-parametric monotonic rank correlations across continuous features.

In [ ]:
corr = df[num_cols + ['readmitted_30d']].corr(method='spearman')
print('Top Spearman correlations with 30-day readmission:')
print(corr['readmitted_30d'].sort_values(ascending=False))

### 8. Generated Figures Verification
List all generated EDA publication-quality figures saved in `reports/figures/`.

In [ ]:
figures = sorted(list(REPORTS_FIGURES_DIR.glob('03_*.png')))
print(f'Total Phase 3 figures generated: {len(figures)}')
for f in figures:
    print(f' - {f.name} ({f.stat().st_size / 1024:.1f} KB)')